### FCC Processing Workflow for the state of California (CA)

In [1]:
### Importing required libraries
import os
import pandas as pd

In [2]:
### Reading all the CA CSV files into a DataFrame
path = '../data/CA/'
ca_data = pd.DataFrame()
for file in os.listdir(path):
    if file.endswith('.csv'):
        df = pd.read_csv(os.path.join(path, file))
        ca_data = pd.concat([ca_data, df], ignore_index=True)
ca_data.head()

,frn,provider_id,brand_name,location_id,technology,max_advertised_download_speed,max_advertised_upload_speed,low_latency,business_residential_code,state_usps,block_geoid,h3_res8_id
0,18735928,131173,AUDEAMUS,1332781653,50,200,100,1,X,CA,60190082007019,8829a95b55fffff
1,7494776,130335,Fidium Fiber,1350860893,50,2000,2000,1,R,CA,60610209083005,882832ab0dfffff
2,7494776,130335,Fidium Fiber,1350864715,50,2000,2000,1,R,CA,60610211091015,88283214e3fffff
3,7494776,130335,Fidium Fiber,1350882146,50,2000,2000,1,R,CA,60610210431018,882832ab39fffff
4,7494776,130335,Fidium Fiber,1350884836,50,2000,2000,1,R,CA,60610210341006,882832ab35fffff


In [3]:
### Running basic data checks
print(ca_data.info())

### Checking for missing values
print(ca_data.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 19402734 entries, 0 to 19402733
Data columns (total 12 columns):
 #   Column                         Dtype
---  ------                         -----
 0   frn                            int64
 1   provider_id                    int64
 2   brand_name                     str  
 3   location_id                    int64
 4   technology                     int64
 5   max_advertised_download_speed  int64
 6   max_advertised_upload_speed    int64
 7   low_latency                    int64
 8   business_residential_code      str  
 9   state_usps                     str  
 10  block_geoid                    int64
 11  h3_res8_id                     str  
dtypes: int64(8), str(4)
memory usage: 1.7 GB
None
frn                              0
provider_id                      0
brand_name                       0
location_id                      0
technology                       0
max_advertised_download_speed    0
max_advertised_upload_speed      0
low_latency 

In [4]:
### Checking unique values for categorical columns
print(ca_data['technology'].unique())
print(ca_data['provider_id'].unique().shape)

[50 40 10]
(90,)


In [5]:
### Reading providers master sheet
providers_master_data = pd.read_csv('../output/final_master_set.csv')
providers_master_data.head()

,Provider ID,FRN,holding_company
0,240058,12609,Mid-Hudson Cablevision
1,240068,12781,"Neu Ventures, Inc."
2,131087,13722,"Rainbow Telecommunications Association, Inc."
3,240058,14936,Mid-Hudson Cablevision
4,131060,17640,"TPC ACC Acquisition, LLC"


In [6]:
### Renaming columns for merging
providers_master_data.rename(columns={'Provider ID': 'provider_id', 'FRN':'frn'}, inplace=True)

In [11]:
### Merging CA data with providers master data to get provider names
merged_data = pd.merge(ca_data, providers_master_data, on=['provider_id', 'frn'], how='left')

In [10]:
### Printing out both the shape of the merged data and the shape of CA data file to check if there are any discrepancies
print("CA Data File Shape:", ca_data.shape)
print("Merged Data Shape:", merged_data.shape)

(19402734, 12)

In [ ]:
### Finding out the counts of Residential and Business customers in the merged data
merged_data['business_residential_code'].value_counts()

business_residential_code
X    11406006
R     7513255
B      483473
Name: count, dtype: int64

### For the purpose of this project we will consider **Residential Addresses Only**

In [14]:
### Filtering out non residential buildings from the merged data
residential_data = merged_data[merged_data['business_residential_code'] == 'R']

In [15]:
### The CB code is a 15-digit code, Adding a padleft to make the necessary leading zeros to the CB code
residential_data['CB_Code'] = residential_data['block_geoid'].apply(lambda x: str(x).zfill(15))
residential_data.head()

,frn,provider_id,brand_name,location_id,technology,max_advertised_download_speed,max_advertised_upload_speed,low_latency,business_residential_code,state_usps,block_geoid,h3_res8_id,holding_company,CB_Code
1,7494776,130335,Fidium Fiber,1350860893,50,2000,2000,1,R,CA,60610209083005,882832ab0dfffff,"Fidium Holdings, LLC",060610209083005
2,7494776,130335,Fidium Fiber,1350864715,50,2000,2000,1,R,CA,60610211091015,88283214e3fffff,"Fidium Holdings, LLC",060610211091015
3,7494776,130335,Fidium Fiber,1350882146,50,2000,2000,1,R,CA,60610210431018,882832ab39fffff,"Fidium Holdings, LLC",060610210431018
4,7494776,130335,Fidium Fiber,1350884836,50,2000,2000,1,R,CA,60610210341006,882832ab35fffff,"Fidium Holdings, LLC",060610210341006
5,7494776,130335,Fidium Fiber,1350888891,50,2000,2000,1,R,CA,60610224001013,88283214c7fffff,"Fidium Holdings, LLC",060610224001013
